# Notebook 05 — Live Demo App (Kaggle T4 + ngrok)

Launches the Gradio dual-mode CXR app on Kaggle GPU with a public ngrok URL.

## Required Inputs:
1. **Kaggle Dataset**: `raddar/chest-xrays-indiana-university` (the images)
2. **Notebook output**: `01+02 - Data + QA + Indexes` (has the indexes)

## Required Secrets:
- `HF_TOKEN` — HuggingFace token with MedGemma access
- `NGROK_TOKEN` — Free from https://dashboard.ngrok.com

## Step 1: huggingface_hub version

In [1]:
!pip install -q --upgrade --force-reinstall huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 79.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Step 2: Install all dependencies

In [2]:
!pip install -q --upgrade peft transformers
!pip install -q gradio pyngrok colpali-engine accelerate bitsandbytes
!pip install -q open-clip-torch faiss-cpu
!pip install -q --upgrade torchao
print('✓ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 16.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.0 MB/s eta 0:00:00a 0:00:01
✓ All packages installed


## Step 3: Configure paths

In [12]:
import os, sys, glob

# Edit these paths to match your Kaggle Input mounts
CORPUS_PATH       = '/kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv'
COLPALI_INDEX_DIR = '/kaggle/input/notebooks/mohammedtaha778/05-app/colpali_index'
CLIP_INDEX_DIR    = '/kaggle/input/notebooks/mohammedtaha778/05-app/clip_index'

# Verify
for name, path in [('corpus', CORPUS_PATH), ('colpali', COLPALI_INDEX_DIR), ('clip', CLIP_INDEX_DIR)]:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'{exists} {name}: {path}')

✓ corpus: /kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv
✓ colpali: /kaggle/input/notebooks/mohammedtaha778/05-app/colpali_index
✓ clip: /kaggle/input/notebooks/mohammedtaha778/05-app/clip_index


## Step 4: Clone repo and set up environment

In [8]:
import subprocess
from kaggle_secrets import UserSecretsClient

WORKING_DIR = '/kaggle/working'
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')

# Clone repo
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)

# Load secrets into environment
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
NGROK_TOKEN = secrets.get_secret('NGROK_TOKEN')

print('✓ Repo ready')
print('✓ Secrets loaded')

✓ Repo ready
✓ Secrets loaded


## Step 5: Load the Gradio app with local indexes

In [9]:
import importlib.util

# Load the app module
spec = importlib.util.spec_from_file_location(
    'app_gradio', 
    os.path.join(REPO_PATH, 'app', 'app_gradio.py')
)
app_mod = importlib.util.module_from_spec(spec)

import importlib, types
import pandas as pd

# Pre-populate module-level vars to skip HF download
app_mod_dict = {
    '__name__': 'app_gradio',
    '__file__': spec.origin,
}

# Execute the module
spec.loader.exec_module(app_mod)

# Override the global vars to use local files instead of downloaded
app_mod.INDEX_DIR = os.path.dirname(COLPALI_INDEX_DIR)
app_mod.path_to_impression = dict(zip(pd.read_csv(CORPUS_PATH)['image_path'], pd.read_csv(CORPUS_PATH)['impression']))

# Override the retriever loaders to use the local paths
def _get_colpali_local():
    if app_mod._colpali is None:
        from src.retrieval.colpali_retriever import ColPaliRetriever
        app_mod._colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)
    return app_mod._colpali

def _get_clip_local():
    if app_mod._clip is None:
        from src.retrieval.clip_retriever import CLIPRetriever
        app_mod._clip = CLIPRetriever()
        app_mod._clip.load_index(CLIP_INDEX_DIR)
    return app_mod._clip

app_mod.get_colpali = _get_colpali_local
app_mod.get_clip = _get_clip_local

print('✓ App loaded with local index paths')

colpali_index/colpali_embeddings.pt:   0%|          | 0.00/964M [00:00<?, ?B/s]

colpali_index/colpali_paths.pkl:   0%|          | 0.00/416k [00:00<?, ?B/s]

clip_index/clip_faiss.index:   0%|          | 0.00/11.2M [00:00<?, ?B/s]

clip_index/clip_paths.pkl:   0%|          | 0.00/416k [00:00<?, ?B/s]

reports_corpus.csv: 0.00B [00:00, ?B/s]

✓ Indexes downloaded
✓ App loaded with local index paths


## Step 6: Launch Gradio + create ngrok tunnel

In [10]:
import threading
import time
from pyngrok import ngrok

# Launch Gradio in background thread
def run_gradio():
    app_mod.demo.launch(server_port=7860, share=False, server_name='0.0.0.0', quiet=True)

gradio_thread = threading.Thread(target=run_gradio, daemon=True)
gradio_thread.start()

print('Starting Gradio... (~20 sec)')
time.sleep(20)

# Setup ngrok
ngrok.kill()  # Kill any existing tunnels
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(7860)

print('\n' + '=' * 60)
print(f'✓ PUBLIC URL: {public_url}')
print('=' * 60)
print('\nOpen this URL in a new browser tab.')
print('First request will be slow (~60s — models loading).')
print('Subsequent requests fast (~5-10s).')
print('\nKeep this notebook running for the URL to stay alive.')

Starting Gradio... (~20 sec)


                                                                                                    
✓ PUBLIC URL: NgrokTunnel: "https://5e0b-34-19-67-19.ngrok-free.app" -> "http://localhost:7860"

Open this URL in a new browser tab.
First request will be slow (~60s — models loading).
Subsequent requests fast (~5-10s).

Keep this notebook running for the URL to stay alive.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

[transformers] It is a prefill stage but The `token_type_ids` is not provided. We recommend passing `token_type_ids` to the model to prevent bad attention masking.


Index loaded: torch.Size([3652, 1031, 128])


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


## Step 7 (optional): Stop the app

Run this when done with the demo:

In [11]:
# Kill ngrok and gradio
ngrok.kill()
app_mod.demo.close()
print('✓ Demo stopped')

Closing server running on port: 7860
✓ Demo stopped
